# Motivation Experiment — Symbolic vs. Semantic guidance under growing context

Run the cells **in order**. Every long-running cell is **resumable**: safe to re-run after a
Colab disconnect (it skips already-completed work). See `RUNBOOK.md` for what each go/no-go
gate looks like and what to do on each failure mode.

**Hard constraints:** T4 GPU only, fp16 everywhere (Turing SM 7.5: no bf16, no FlashAttention-2).


## Cell 1 — Setup (deps, repos, auth)
Pins deps **without touching Colab's preinstalled torch** (a torch swap breaks T4 CUDA).

In [ ]:

# --- Config you MUST set ---
SELF_REPO   = "https://github.com/PLACEHOLDER/dag-motivation-exp"  # <-- your repo
SELF_COMMIT = "main"                                              # <-- pin a commit for reproducibility
PRONTOQA_COMMIT = "PLACEHOLDER_PIN_ME"                            # <-- pin github.com/asaparov/prontoqa commit
DRIVE_DIR   = "/content/drive/MyDrive/motivation_exp"            # outputs live here (survive disconnects)

import subprocess, sys, os

# 1) Mount Drive (outputs + resume state persist here)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

# 2) Pin torch to the EXACT preinstalled version via a constraints file so nothing swaps it.
import torch
torch_ver = torch.__version__.split("+")[0]
with open("/content/constraints.txt", "w") as f:
    f.write(f"torch=={torch_ver}\n")
print("Colab torch:", torch.__version__, "| CUDA:", torch.version.cuda)

# 3) Clone + install the package (core is torch-free; [colab] extra adds xgrammar + s-t,
#    installed under the constraints file so pip cannot replace torch).
if not os.path.isdir("/content/dag-motivation-exp"):
    subprocess.run(["git", "clone", SELF_REPO, "/content/dag-motivation-exp"], check=True)
subprocess.run(["git", "-C", "/content/dag-motivation-exp", "checkout", SELF_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/content/constraints.txt",
                "-e", "/content/dag-motivation-exp[colab]"], check=True)

# 4) Fetch the pinned PrOntoQA generator.
subprocess.run(["bash", "/content/dag-motivation-exp/scripts/fetch_prontoqa.sh",
                "/content/prontoqa", PRONTOQA_COMMIT], check=True)

# 5) Hugging Face auth (Llama-3.2-3B is gated).
from huggingface_hub import login
login()  # paste a token with access to meta-llama/Llama-3.2-3B-Instruct

# 6) Sanity: nothing swapped torch.
import torch as _t; assert _t.__version__.split("+")[0] == torch_ver, "torch was changed by an install!"
print("Setup OK. torch intact:", _t.__version__)

## Cell 2 — Phase 0: Environment & smoke test
GPU assert, fp16 load, 8k generation with `logits_to_keep=1`, **peak-VRAM check**, eot-token assert.

In [ ]:

import torch, time, json, hashlib, subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
from motivation_exp import config as C

# 1) T4 assert (all latency numbers must come from T4).
name = torch.cuda.get_device_name(0)
assert "T4" in name, f"Not a T4 (got {name!r}). Disconnect + retry (Runtime > Disconnect and delete runtime)."
print("GPU:", name)

# 2) Load model fp16 + SDPA (no bf16 / no FA2 on Turing).
tok = AutoTokenizer.from_pretrained(C.PRIMARY_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    C.PRIMARY_MODEL, torch_dtype=torch.float16, attn_implementation="sdpa", device_map="cuda",
)
model.eval()
print("vocab_size:", model.config.vocab_size)

# 3) Stop-token assert (Llama-3.2-Instruct stops on <|eot_id|>, NOT <|end_of_text|>).
eot_id = tok.convert_tokens_to_ids(C.STOP_TOKEN)
assert tok.convert_ids_to_tokens(eot_id) == C.STOP_TOKEN, "eot id mismatch"
print("eot id:", eot_id, "=", C.STOP_TOKEN)

# 4) Embedder fp16 on GPU.
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(C.EMBED_MODEL, device="cuda",
                               model_kwargs={"torch_dtype": torch.float16})
print("embedder loaded:", C.EMBED_MODEL)

# 5) 8k-context generation with logits_to_keep=1 + peak-VRAM check (rev #1).
torch.cuda.reset_peak_memory_stats()
filler = ("Every tumpus is a wumpus. " * 2000)
ids = tok(filler, return_tensors="pt").input_ids[:, :8192].cuda()
cache = DynamicCache()
torch.cuda.synchronize(); t0 = time.perf_counter()
with torch.inference_mode():
    out = model(input_ids=ids, attention_mask=torch.ones_like(ids),
                past_key_values=cache, use_cache=True, logits_to_keep=1)
    for _ in range(8):  # a few decode steps
        nid = out.logits[0, -1].argmax().view(1, 1)
        attn = torch.ones((1, cache.get_seq_length() + 1), device="cuda", dtype=torch.long)
        out = model(input_ids=nid, attention_mask=attn, past_key_values=cache,
                    use_cache=True, logits_to_keep=1)
torch.cuda.synchronize()
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"8k smoke OK in {time.perf_counter()-t0:.1f}s | peak VRAM {peak_gb:.2f} GB (must be < ~15 GB)")
assert peak_gb < 15.5, "Too close to the 16 GB limit; cap the top bucket at 6k (see RUNBOOK)."

# 6) Env manifest.
freeze = subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout
env = {"gpu": name, "torch": torch.__version__, "cuda": torch.version.cuda,
       "vocab_size": model.config.vocab_size, "eot_id": eot_id, "peak_vram_gb_8k": peak_gb}
env["env_hash"] = hashlib.sha1(freeze.encode()).hexdigest()[:12]
with open(f"{DRIVE_DIR}/env_manifest.json", "w") as f: json.dump(env, f, indent=2)
with open(f"{DRIVE_DIR}/pip_freeze.txt", "w") as f: f.write(freeze)
print("env_hash:", env["env_hash"])

## Cell 3 — Phase 1: Data generation
Generates base questions with the PrOntoQA generator, adapts them defensively, builds the
distractor pool, reserves the calibration + exemplar splits (disjoint from all buckets),
buckets to token targets, runs the validation gate, and writes `items.jsonl` + `manifest.json`.

**If this cell errors in `raw_prontoqa_adapter`,** the upstream return shape differs from what
the adapter expects — print one raw item (see the commented line) and extend the adapter's
key lists in `datagen.py`. This is the one place we could not verify without the repo cloned.

In [ ]:

import sys, json, os, random
sys.path.insert(0, "/content/prontoqa")   # make the cloned generator importable
from motivation_exp import datagen as dg, config as C

# ---- Adapter onto the generator. VERIFY/ADJUST the call below against the cloned repo. ----
import importlib
gen_mod = importlib.import_module("run_experiment")   # prontoqa's entry module

def generate_base_items(n):
    """Call PrOntoQA's generate_question n times; adapt each into an Item.
    NOTE: signature/kwargs may differ across commits — adjust to the cloned version.
    """
    items = []
    rng = random.Random(C.GLOBAL_SEED)
    for i in range(n):
        raw = gen_mod.generate_question(
            num_deduction_steps=C.N_HOPS, ontology=C.ONTOLOGY,   # <-- adjust kwargs if needed
        )
        # print(raw); raise SystemExit  # <-- UNCOMMENT once to inspect the raw shape, then adapt.
        items.append(dg.raw_prontoqa_adapter(raw, item_id=f"base{i:04d}", base_id=f"base{i:04d}"))
    return items

N_BASE = 150   # 100 for full + spares that may fail validation
bases = generate_base_items(N_BASE)

# Distractor pool from independent ontologies (generate extra items just for their sentences).
pool_items = generate_base_items(200)
pool = dg.build_distractor_pool(pool_items, size=20000, seed=C.GLOBAL_SEED)

# Reserve disjoint splits BEFORE bucketing (R2 #1/#2).
bucket_bases, calib, exemplar = dg.reserve_splits(bases, C.N_CALIB_ITEMS, C.N_EXEMPLAR_ITEMS)

# Bucket per the Llama tokenizer.
count_tokens = dg.hf_token_counter(tok)
buckets = dg.bucket_to_token_targets(bucket_bases, count_tokens, pool, C.BUCKETS, C.GLOBAL_SEED)

# Validation gate.
kept = {b: [] for b in C.BUCKETS}
dropped = 0
for b, its in buckets.items():
    for it in its:
        ok, reason = dg.validate_item(it, count_tokens)
        if ok: kept[b].append(it)
        else: dropped += 1; print("DROP", it.item_id, reason)
print("kept per bucket:", {b: len(v) for b, v in kept.items()}, "| dropped:", dropped)

# Persist to Drive.
data_dir = f"{DRIVE_DIR}/data"; os.makedirs(data_dir, exist_ok=True)
for b, its in kept.items():
    os.makedirs(f"{data_dir}/{b}", exist_ok=True)
    with open(f"{data_dir}/{b}/items.jsonl", "w") as f:
        for it in its: f.write(json.dumps(it.to_json()) + "\n")
with open(f"{data_dir}/calib.jsonl", "w") as f:
    for it in calib: f.write(json.dumps(it.to_json()) + "\n")
with open(f"{data_dir}/exemplar.jsonl", "w") as f:
    for it in exemplar: f.write(json.dumps(it.to_json()) + "\n")
dg.write_manifest(kept, f"{data_dir}/manifest.json", C.GLOBAL_SEED,
                  calib_ids=[c.base_id for c in calib],
                  exemplar_ids=[e.base_id for e in exemplar])
print("Phase 1 done ->", data_dir)

## Cell 4 — Phase 1.5: τ calibration (freeze thresholds)
Sweeps `TAU_GRID` on the 10 held-out calibration items. Positives = gold proof steps
(should be accepted); negatives = concept-swapped corruptions (should be rejected). Picks the
threshold with the best balanced accuracy and **prints the value to paste into `config.py`**.
No reported run may use uncalibrated or re-tuned τ (R2 #1).

In [ ]:

import numpy as np, random
from motivation_exp.checker import SemanticChecker, sentence_transformer_encoder
from motivation_exp import config as C

encode_fn = sentence_transformer_encoder(embedder)
rng = random.Random(C.GLOBAL_SEED)

def corrupt(step, concepts):
    """Swap the concept in a gold step for a different in-pool concept -> should be rejected."""
    for c in concepts:
        if c in step:
            other = rng.choice([x for x in concepts if x != c] or [c])
            return step.replace(c, other, 1)
    return step + " zzzz."

pos_cos, neg_cos = [], []
all_concepts = sorted({c for it in calib for c in it.concepts})
for it in calib:
    chk = SemanticChecker(encode_fn, tau_restate=2.0, tau_mp=2.0)  # tau=2 => never auto-accepts
    chk.prefill(it.context)
    for step in it.gold_steps:
        pos_cos.append(chk.check_step(step).cosine)
        chk.accept(step)   # grow candidate set as the proof proceeds
        neg_cos.append(chk.check_step(corrupt(step, all_concepts)).cosine)

pos_cos, neg_cos = np.array(pos_cos), np.array(neg_cos)
print(f"positives n={len(pos_cos)} mean={pos_cos.mean():.3f} | negatives n={len(neg_cos)} mean={neg_cos.mean():.3f}")
best = None
for tau in C.TAU_GRID:
    tpr = float((pos_cos >= tau).mean()); tnr = float((neg_cos < tau).mean())
    bal = 0.5 * (tpr + tnr)
    print(f"  tau={tau:.2f}  TPR(gold accept)={tpr:.2f}  TNR(corrupt reject)={tnr:.2f}  bal={bal:.3f}")
    if best is None or bal > best[1]: best = (tau, bal)
TAU = best[0]
print(f"\n>>> SELECTED tau = {TAU}. Paste into config.py:  TAU_RESTATE = {TAU}   TAU_MP = {TAU}")
# also record in the manifest
import json
mpath = f"{DRIVE_DIR}/data/manifest.json"; m = json.load(open(mpath))
m["frozen_tau"] = {"tau_restate": TAU, "tau_mp": TAU}; json.dump(m, open(mpath, "w"), indent=2)

## Cell 5 — Phase 2: Pilot (20 items/bucket) + go/no-go gates
Runs all three methods interleaved. Resumable. Prints G1/G2/G3. **Freeze τ in `config.py`
(or set it below) before running** — the checker refuses uncalibrated τ.

In [ ]:

import json, itertools
from motivation_exp import runner as R, config as C
from motivation_exp.grammar import make_compiler, synthesize_exemplar_cot
from motivation_exp.checker import SemanticChecker, sentence_transformer_encoder

# --- set the frozen tau (paste from Phase 1.5 if not yet edited into config.py) ---
TAU_RESTATE = C.TAU_RESTATE if C.TAU_RESTATE is not None else TAU
TAU_MP      = C.TAU_MP      if C.TAU_MP      is not None else TAU

encode_fn = sentence_transformer_encoder(embedder)
compiler  = make_compiler(tok, model.config.vocab_size)
exemplar_item = exemplar[0]
exemplar_cot  = synthesize_exemplar_cot(exemplar_item)
cfg = R.DecodeCfg(eot_id=eot_id, max_new_tokens=C.MAX_NEW_TOKENS,
                  per_step_cap=C.PER_STEP_TOKEN_CAP, temp_schedule=tuple(C.TEMPERATURE_SCHEDULE),
                  max_retries=C.SEMANTIC_MAX_RETRIES)
sync, sample_fn = R.make_torch_helpers()
model_name = C.MODEL_SHORT[C.PRIMARY_MODEL]
env = json.load(open(f"{DRIVE_DIR}/env_manifest.json"))

def checker_factory():
    return SemanticChecker(encode_fn, TAU_RESTATE, TAU_MP)

# take the first 20 items/bucket -> these are also the first 20 of the full run (rev #10)
pilot = {b: v[:C.PILOT_ITEMS_PER_BUCKET] for b, v in kept.items()}
pilot_out = f"{DRIVE_DIR}/results_pilot.jsonl"
R.run_all(model, tok, pilot, C.METHODS, pilot_out, model_name,
          exemplar_item=exemplar_item, exemplar_cot=exemplar_cot, cfg=cfg, sync=sync,
          sample_fn=sample_fn, compiler=compiler, checker_factory=checker_factory,
          gpu=env["gpu"], env_hash=env["env_hash"])

# ---- gates ----
from motivation_exp import plots
df = plots.load_results(pilot_out)
acc = plots._accuracy_table(df); lat = plots._latency_table(df)
g1 = acc.get("unguided", {}).get(min(C.BUCKETS), (0,))[0]
print(f"G1 unguided@{min(C.BUCKETS)} acc = {g1:.2f}  (want 0.40-0.80; <0.40 raise Qwen/hops=2, >0.80 hops=4)")
print("G2 (semantic >= symbolic, gap non-shrinking):")
for b in sorted(C.BUCKETS):
    s = acc.get("semantic", {}).get(b, (float('nan'),))[0]; y = acc.get("symbolic", {}).get(b, (float('nan'),))[0]
    print(f"   bucket {b}: semantic={s:.2f} symbolic={y:.2f} gap={s-y:+.2f}")
print("G3 (semantic latency rising, symbolic flat):")
for b in sorted(C.BUCKETS):
    s = lat.get("semantic", {}).get(b, (float('nan'),))[0]; y = lat.get("symbolic", {}).get(b, (float('nan'),))[0]
    print(f"   bucket {b}: semantic={s:.1f}ms symbolic={y:.1f}ms")

## Cell 6 — Phase 3: Full run (100 items/bucket, resumable)
Shares its output file with the pilot **by design (rev #10)**: the pilot's 20 completed
(model, method, bucket, item_id) tuples are skipped, not recomputed. Re-run this cell after
any disconnect until it prints `ALL DONE`.

In [ ]:

from motivation_exp import runner as R, config as C

full = {b: v[:C.FULL_ITEMS_PER_BUCKET] for b, v in kept.items()}
full_out = f"{DRIVE_DIR}/results_full.jsonl"

# One-time: copy pilot rows into the full file so they count as completed (same items/seeds).
import os, shutil
if not os.path.exists(full_out) and os.path.exists(pilot_out):
    shutil.copyfile(pilot_out, full_out)

R.run_all(model, tok, full, C.METHODS, full_out, model_name,
          exemplar_item=exemplar_item, exemplar_cot=exemplar_cot, cfg=cfg, sync=sync,
          sample_fn=sample_fn, compiler=compiler, checker_factory=checker_factory,
          gpu=env["gpu"], env_hash=env["env_hash"])

# completion check
from motivation_exp import plots
done = plots.load_results(full_out)
need = len(C.BUCKETS) * C.FULL_ITEMS_PER_BUCKET * len(C.METHODS)
print(f"rows: {len(done)} / {need}")
print("ALL DONE" if len(done) >= need else "NOT DONE — re-run this cell after reconnecting.")

## Cell 7 — Phase 4: Two-panel motivation figure
Panel A accuracy (Wilson CIs), Panel B median per-token latency (IQR band). Exports PNG + PDF.

In [ ]:

from motivation_exp import plots, config as C
df = plots.load_results(f"{DRIVE_DIR}/results_full.jsonl")
out_png = f"{DRIVE_DIR}/motivation_figure.png"
plots.make_two_panel_figure(df, out_png, model_name=C.MODEL_SHORT[C.PRIMARY_MODEL])
print("wrote", out_png, "and .pdf")
from IPython.display import Image, display
display(Image(out_png))